## Pulling the bill data
JOKES can't do econ/recession stuff. Probably for the best. Going to keep econ indicators as controls (like COVID inflation/unemployment, etc.)

### TO DO
* Assign yearly unemployment, state GDP, general fund growth (maybe if I can find it; might be better to get income).
* Get gender and party data for 2026 legislators; make bipartisan variable
* Explore compiled bill data; how many bills in each subject, avg number of cosponsors, get idea of what to do.

### KEY VARIABLES
#### SIMPLE BILL MODEL
* Number of cosponsors
* Money appropriated sum OR if money was appropriated
* Number of substitutes
* Subject area (want to narrow from 21 to reduce multicolin)
* Year fixed effect (COVID?)

#### 2026 ONLY MODEL
* Gender
* Party (might not be worth doing; UT overwhelming republican)
* Bipartisan

#### ECON CONTROLLED MODEL
* Unemployment
* GDP
* General Fund Growth

In [ ]:
# importing packages
import requests
import pandas as pd
import json
import time
import numpy as np
import re

In [ ]:
# pulling data from the legislative API [provided by Lucy Ballou]

TOKEN = "redacted"
BASE_URL = "https://glen.le.utah.gov/bills"

# defining sessions (pre-2016 is unavail with API)
sessions = [
    "2016GS", "2017GS", "2018GS", "2019GS", "2020GS",
    "2021GS", "2022GS", "2023GS", "2024GS", "2025GS", "2026GS"
]

# extracting bill numbers
all_bills = {}

for session in sessions:
    url = f"{BASE_URL}/{session}/billlist/{TOKEN}"
    response = requests.get(url)

    if response.status_code == 200:
        try:
            data = response.json()

            # handling the fact that some years are formatted as plain lists, some are objects, some are strings
            if isinstance(data, list):
                bill_list = data
            elif isinstance(data, dict) and "bills" in data:
                bill_list = data["bills"]
            else:
                print(f"{session}: Unexpected format")
                continue

            bill_numbers = [bill['number'] for bill in bill_list]
            all_bills[session] = bill_numbers
            print(f"{session}: {len(bill_numbers)} bills")
        except Exception as e:
            print(f"{session}: Failed - {e}")
    else:
        print(f"{session}: ERROR {response.status_code}")

    time.sleep(1)

total = sum(len(bills) for bills in all_bills.values())
print(f"\nTotal bills across sessions: {total}")

# actual extracting
def extract_bill_data(session, bill_number, token):
    url = f"{BASE_URL}/{session}/{bill_number}/{token}"
    response = requests.get(url)

    if response.status_code != 200:
        return None

    try:
        data = response.json()
    except:
        return None

    if isinstance(data, dict) and "bill" in data and isinstance(data["bill"], dict):
        data = data["bill"]

    if isinstance(data, str):
        try:
            data = json.loads(data)
        except:
            return None

    # skip if still not a dict
    if not isinstance(data, dict):
        return None

    is_old_format = "shorttitle" in data

    if is_old_format:
        # old format
        subjects = data.get("subjects", [])

        # committees from agendas
        agendas = data.get("agendas", [])
        committees = list(set(
            a.get("committee", a.get("committeeName", ""))
            for a in agendas
        )) if agendas else []

        # originating chamber from bill number prefix
        bill_id = data.get("bill", "")
        if bill_id.startswith("HB") or bill_id.startswith("HJ"):
            orig_chamber = "House"
        elif bill_id.startswith("SB") or bill_id.startswith("SJ"):
            orig_chamber = "Senate"
        else:
            orig_chamber = ""

        return {
            "bill_id": bill_id,
            "short_title": data.get("shorttitle", ""),
            "sponsor": data.get("sponsor", ""),
            "co_sponsors": "None",
            "floor_sponsor": data.get("floorsponsor", ""),
            "last_action": data.get("lastaction", ""),
            "last_action_owner": data.get("lastactionowner", ""),
            "subjects": "; ".join(subjects) if isinstance(subjects, list) else str(subjects),
            "num_substitutes": 0,
            "substitute_sponsors": "None",
            "active_version": data.get("version", ""),
            "subjects_changed": False,
            "committees": "; ".join(committees),
            "money_appropriated": data.get("monies", ""),
            "originating_chamber": orig_chamber,
            "final_chamber": "",
            "sponsor_id": data.get("sponsor", ""),
            "session_year": session[:4],
            "session_id": session,
        }

    else:
        # new format
        versions = data.get("billVersionList", [])

        # finding and extracting active version for co-sponsors and subjects
        active_version = None
        active_version_label = ""
        for version in versions:
            if version.get("activeVersion", False):
                active_version = version
                active_version_label = version.get("billNumber", "")
                break
        if active_version is None and versions:
            active_version = versions[-1]
            active_version_label = versions[-1].get("billNumber", "")

        subjects = []
        if active_version:
            subjects = list(set(
                s["description"] for s in active_version.get("subjectList", [])
            ))

        co_sponsors = []
        if active_version:
            co_sponsors = [
                cs.get("name", "") for cs in active_version.get("coSponsorList", [])
            ]

        # substitute stuff
        substitutes = [v for v in versions if int(v.get("subVersion", 0)) > 0]
        num_substitutes = len(substitutes)

        sub_sponsors = "; ".join(
            f"{v['billNumber']}: {v.get('versionSponsorName', 'Unknown')}"
            for v in substitutes
        )

        # whether subjects changed across versions?
        def get_subjects_set(version):
            return set(s["description"] for s in version.get("subjectList", []))

        if len(versions) > 1:
            original_subjects = get_subjects_set(versions[0])
            subjects_changed = any(
                get_subjects_set(v) != original_subjects for v in versions[1:]
            )
        else:
            subjects_changed = False

        # getting committees from agendaList
        committees = list(set(
            a["committeeName"] for a in data.get("agendaList", [])
        ))

        # finding original and final chamber from action history
        originating_chamber = data.get("primeSponsorHouse", "")
        final_chamber = ""
        for action in reversed(data.get("actionHistoryList", [])):
            if action.get("actionClass") in ("H", "S"):
                final_chamber = "House" if action["actionClass"] == "H" else "Senate"
                break

        return {
            "bill_id": data.get("billNumber", ""),
            "short_title": data.get("shortTitle", ""),
            "sponsor": data.get("primeSponsorName", ""),
            "co_sponsors": "; ".join(co_sponsors) if co_sponsors else "None",
            "floor_sponsor": data.get("floorSponsorName", ""),
            "last_action": data.get("lastAction", ""),
            "last_action_owner": data.get("lastActionOwner", ""),
            "subjects": "; ".join(subjects),
            "num_substitutes": num_substitutes,
            "substitute_sponsors": sub_sponsors if sub_sponsors else "None",
            "active_version": active_version_label,
            "subjects_changed": subjects_changed,
            "committees": "; ".join(committees),
            "money_appropriated": data.get("moniesAppropriated", ""),
            "originating_chamber": "House" if originating_chamber == "H" else "Senate",
            "final_chamber": final_chamber,
            "sponsor_id": data.get("primeSponsor", ""),
            "session_year": data.get("year", ""),
            "session_id": data.get("sessionID", ""),
        }


# extraction loop
all_rows = []

for session, bill_numbers in all_bills.items():
    print(f"\n--- {session}: {len(bill_numbers)} bills ---")

    for i, bill_number in enumerate(bill_numbers):
        result = extract_bill_data(session, bill_number, TOKEN)

        if result:
            all_rows.append(result)
        else:
            print(f"  Failed: {bill_number}")

        # progress update every 50 bills
        if (i + 1) % 50 == 0:
            print(f"  Processed {i + 1}/{len(bill_numbers)}")

        time.sleep(1)

    # save progress after each session
    df_progress = pd.DataFrame(all_rows)
    df_progress.to_csv("utah_bills_progress.csv", index=False)
    print(f"  Saved progress: {len(all_rows)} total rows so far")

# build dataframe
df = pd.DataFrame(all_rows)
df.to_csv("utah_bills_final.csv", index=False)
print(f"\nFinal dataset: {df.shape[0]} rows, {df.shape[1]} columns")
print(df.head())


In [ ]:
# removing all resolutions (simple, joint, concurrent) to have dataset of only laws
laws = ['SJR', 'HJR', 'SCR', 'HCR', 'SR0', 'HR0']
df_bills = ut_bills.loc[~ut_bills['bill_id'].str[:3].isin(laws)].copy().reset_index(drop=True)
print(df_bills.shape, ut_bills.shape)




# cleaning NAs of subjects, committees, cosponsors, floor_sponsor, money_approrpiated
na_names = ['subjects', 'committees','co_sponsors', 'floor_sponsor', 'money_appropriated']
df_bills[na_names] = df_bills[na_names].replace({np.nan:'None'})




# making lists of all action types

location0 = ['House/ filed', 'Senate/ filed', ''] # died in starting chamber (committee, house or senate)

location1 = ['House/ received from Senate', 
            'Senate/ received from House',
            'Senate/ 2nd Reading Calendar to Rules'] # died in non-originating chamber

location2 = ['Governor Line Item Veto',
            'Governor Vetoed',
            'Senate/ strike enacting clause',
            'Bill Failed Review - Returned to House'] # died in gov office (or post-Legislature)

location3 = ['LFA/ fiscal note publicly available',
            'LFA/ fiscal note sent to sponsor',
            'Became Law w/o Governor Signature',
            'Senate/ to Governor',
            'House/ to Governor',
            'House/ to Lieutenant Governor',
            'Senate/ to Lieutenant Governor',
            'Governor Declined to Sign',
            'Senate/ received enrolled bill from Printing',
            'House/ enrolled bill to Printing',
            'Senate/ enrolled bill to Printing',
            'Governor Signed'] # final location (aka passed)

# converting 'last_action' to ordinal range 'final_location' (god bless numpy select)
conditions = [df_bills['last_action'].isin(location0),
              df_bills['last_action'].isin(location1),
              df_bills['last_action'].isin(location2),
              df_bills['last_action'].isin(location3)]
choices = [0,1,2,3]

df_bills['location_stage'] = np.select(conditions, choices)

# converting 'last_action' to 'passed' binary variable
df_bills['passed'] = df_bills['last_action'].map(lambda x: 1 if x in location3 else 0)




# grouping our 870 subjects (yes, 870) into 21 groups
subject_to_group = {
    # Criminal Justice
    'Crimes': 'crim_justice', 'Offenses': 'crim_justice', 'Criminal Procedure': 'crim_justice',
    'Criminal Code': 'crim_justice', 'Criminal Conduct': 'crim_justice', 'Criminal Identification': 'crim_justice',
    'Sentencing': 'crim_justice', 'Parole': 'crim_justice', 'Probation': 'crim_justice',
    'Expungement': 'crim_justice', 'Juvenile Justice': 'crim_justice', 'Juvenile Justice Services': 'crim_justice',
    'Peace Officers': 'crim_justice', 'Peace Officer': 'crim_justice', 'Courts': 'crim_justice',
    'Arrest and Detention': 'crim_justice', 'Bail\\Pretrial Detention': 'crim_justice',
    'Bail/Pretrial Detention': 'crim_justice', 'Plea Agreements': 'crim_justice',
    'Restitution': 'crim_justice', 'Recidivism': 'crim_justice', 'Homicide': 'crim_justice',
    'Theft': 'crim_justice', 'Assault and Battery': 'crim_justice', 'Vandalism': 'crim_justice',
    'Kidnapping': 'crim_justice', 'Prostitution': 'crim_justice', 'Gambling': 'crim_justice',
    'Gangs': 'crim_justice', 'Human Trafficking': 'crim_justice', 'Sexual Offenses': 'crim_justice',
    'Sex Offender Registry': 'crim_justice', 'Offender Registries': 'crim_justice',
    'Death Penalty': 'crim_justice', 'Insanity Defense': 'crim_justice',
    'Insanity Defense/Competency': 'crim_justice', 'Competency': 'crim_justice',
    'Forfeiture Procedure': 'crim_justice', 'Search and Seizure': 'crim_justice',
    'Protective Orders': 'crim_justice', 'Protective Orders/Stalking Injuctions': 'crim_justice',
    'Bureau of Criminal Identification': 'crim_justice', 'Bureau of Criminal Investigation': 'crim_justice',
    'Commission on Criminal and Juvenile Justice': 'crim_justice', 'Sentencing Commission': 'crim_justice',
    'Inmates': 'crim_justice', 'Correctional Facilities': 'crim_justice',
    'Department of Corrections': 'crim_justice', 'Youth Corrections': 'crim_justice',
    'Juvenile Records': 'crim_justice', 'Code of Criminal Procedure': 'crim_justice',
    'Board of Pardons and Parole': 'crim_justice', 'Indigent Defense': 'crim_justice',
    'Indigent Counsel': 'crim_justice', 'Drug Court': 'crim_justice',
    'Law Enforcement and Criminal Justice': 'crim_justice', 'Peace Officer Standards and Training': 'crim_justice',
    'Utah Highway Patrol': 'crim_justice', 'Fraud': 'crim_justice',
    'Punishments': 'crim_justice', 'Punishment': 'crim_justice',
    'Self Defense': 'crim_justice', 'Defenses': 'crim_justice',
    'Weapons': 'crim_justice', 'Graffiti': 'crim_justice',

    # Education K-12
    'K-12 Education': 'ed_k12', 'School Finance': 'ed_k12', 'School Safety': 'ed_k12',
    'School Districts': 'ed_k12', 'Special Education': 'ed_k12',
    'Standards and Curriculum': 'ed_k12', 'School Accountability': 'ed_k12',
    'School Personnel': 'ed_k12', 'School Fees': 'ed_k12',
    'School Facilities': 'ed_k12', 'Student Assessment': 'ed_k12',
    'Student Discipline': 'ed_k12', 'Student Privacy': 'ed_k12',
    'Student Health and Safety': 'ed_k12', 'Student health and safety': 'ed_k12',
    'Local Boards of Education': 'ed_k12', 'State Board of Education': 'ed_k12',
    'Local Education Agencies (LEAs)': 'ed_k12', 'Charter Schools': 'ed_k12',
    'Home Schooling': 'ed_k12', 'Prekindergarten': 'ed_k12',
    'School Community Councils': 'ed_k12', 'School community councils': 'ed_k12',
    'State School Funding Distribution': 'ed_k12', 'Public Education': 'ed_k12',
    'Public Education Data and Reporting': 'ed_k12', 'Education Grant Programs': 'ed_k12',
    'Competency in Education': 'ed_k12', 'Strategic Planning for Education': 'ed_k12',
    'School and Institutional Trust Fund Office': 'ed_k12',
    'School and Institutional Trust Land Administration': 'ed_k12',
    'Online Learning and/or Technology': 'ed_k12', 'Educational Telecommunications': 'ed_k12',
    'Applied Technology Education': 'ed_k12', 'Technical Colleges': 'ed_k12',
    'Campus Safety': 'ed_k12', 'On-campus Speech': 'ed_k12',

    # Higher Education
    'Higher Education': 'higher_ed', 'Colleges and Universities': 'higher_ed',
    'Utah Board of Higher Education': 'higher_ed', 'State Board of Regents': 'higher_ed',
    'Utah System of Higher Education': 'higher_ed', 'Scholarships and Financial Aid': 'higher_ed',
    'Concurrent Enrollment': 'higher_ed', 'Nontraditional Students': 'higher_ed',
    'Nontraditional students': 'higher_ed', 'Higher Education Students': 'higher_ed',
    'Performance Funding': 'higher_ed', 'Utech Board of Trustees': 'higher_ed',
    'Utah Education Savings Plan': 'higher_ed',

    # Health Care
    'Public Health': 'health_care', 'Mental Health': 'health_care', 'Medicaid': 'health_care',
    'Medicare': 'health_care', 'Health Insurance': 'health_care', 'Hospitals': 'health_care',
    'Telehealth': 'health_care', 'Substance Abuse': 'health_care', 'Substance Abuse and Mental Health': 'health_care',
    'Substance Use Disorder Treatment': 'health_care', 'Opioids': 'health_care',
    'Medical Cannabis': 'health_care', 'Pharmaceuticals': 'health_care', 'Pharmacies': 'health_care',
    'Pharmacists': 'health_care', 'Pharmacy Benefit Managers': 'health_care',
    'Health Care': 'health_care', 'Health Care Facilities': 'health_care',
    'Health Care Professionals': 'health_care', 'Health Care Providers': 'health_care',
    'Health Care Data and Informatics': 'health_care', 'Health Care Statistics': 'health_care',
    'Health Benefits and Claims': 'health_care', 'Benefits/Claims, Health': 'health_care',
    'Health': 'health_care', 'Health and Human Services': 'health_care',
    'Department of Health': 'health_care', 'Department of Health and Human Services': 'health_care',
    'Diseases': 'health_care', 'Disease Control and Prevention': 'health_care',
    'Diseases Control and Prevention': 'health_care', 'Immunizations': 'health_care',
    'Maternal and Child Health': 'health_care', 'Family Health and Preparedness': 'health_care',
    'Mental Health Facilities': 'health_care', 'Mental Health Professional': 'health_care',
    'Mental Health Professionals': 'health_care', 'Local Mental Health Authorities': 'health_care',
    'Local Substance Abuse Authorities': 'health_care', 'Suicide': 'health_care',
    'Nursing Homes': 'health_care', 'Skilled Nursing Facilities': 'health_care',
    'Ambulatory Surgical Centers': 'health_care', 'Community Health Clinics': 'health_care',
    'Utah State Hospital': 'health_care', 'Utah State Developmental Center': 'health_care',
    'Facility Licensing, Certification, and Resident Assessment': 'health_care',
    'Health Facility Administrators': 'health_care', 'Physicians and Surgeons': 'health_care',
    'Osteopathic Physicians and Surgeons': 'health_care', 'Physician Assistants': 'health_care',
    'Nurses': 'health_care', 'Dentists and Dental Hygienists': 'health_care',
    'Optometrists': 'health_care', 'Psychologists': 'health_care',
    'Physical Therapists': 'health_care', 'Occupational Therapists': 'health_care',
    'Speech-Language Pathologists and Audiologists': 'health_care', 'Respiratory Care Providers': 'health_care',
    'Midwives': 'health_care', 'Acupuncturists': 'health_care', 'Naturopathic Physicians': 'health_care',
    'Chiropractic Physicians': 'health_care', 'Podiatric Physicians': 'health_care',
    'Massage Therapists': 'health_care', 'Hearing Instrument Specialists': 'health_care',
    'Radiologic Technologists, Assistants, and Technicians': 'health_care',
    'Dietitians': 'health_care', 'Genetic Counselors': 'health_care',
    'Athletic Trainers': 'health_care', 'Medical Records': 'health_care',
    'Medical Malpractice': 'health_care', 'Medical Examiner': 'health_care',
    'Medical Fees and Service for Employment': 'health_care',
    'Medical Assistance Programs': 'health_care', 'Advance Health Care Directive': 'health_care',
    'Civil Commitment': 'health_care', 'Abortion': 'health_care',
    'Electronic Cigarettes': 'health_care', 'Tobacco': 'health_care',
    'Tobacco and Other Nicotine Products': 'health_care', 'Alcohol': 'health_care',
    'Alcoholic Beverages': 'health_care', 'Alcoholic Beverage Control': 'health_care',
    'Alcoholic Beverage Control Commission': 'health_care',
    'Department of Alcoholic Beverage Control': 'health_care',
    'Department of Alcoholic Beverage Services': 'health_care',
    'Drug and Alcohol Testing': 'health_care', 'Drugs and Alcohol, Workplace': 'health_care',
    'Controlled Substances': 'health_care', 'Controlled Substances Database': 'health_care',
    'Public Health Laboratory': 'health_care', 'Vital Statistics': 'health_care',
    'Emergency Medical Services': 'health_care', 'Emergency Medical Services and Preparedness': 'health_care',
    'Rural Health': 'health_care', 'Children\'s Health Insurance Program': 'health_care',
    'Rehabilitation Services': 'health_care', 'Adult Protective Services': 'health_care',
    'Long-term Care Ombudsman': 'health_care', 'Aging and Adult Services': 'health_care',
    'Aging Services': 'health_care', 'Insurance, Health': 'health_care',
    'Commercial Health Insurance\\Managed Care Contracts': 'health_care',
    'Public Employees\' Health Program': 'health_care',

    # Environment & Natural Resources
    'Air Quality': 'env_nr', 'Water Quality': 'env_nr',
    'Water': 'env_nr', 'Water Rights': 'env_nr',
    'Water Rights - Const. Art. XVII': 'env_nr',
    'Water Conservation': 'env_nr', 'Water and Irrigation': 'env_nr',
    'Water Utilities, Irrigation, and Sewer': 'env_nr',
    'Drinking Water': 'env_nr', 'Great Salt Lake': 'env_nr',
    'Utah Lake': 'env_nr', 'Mining': 'env_nr',
    'Mines and Mining': 'env_nr', 'Forestry and Fire': 'env_nr',
    'Forestry, Fire, and State Lands': 'env_nr',
    'Forests': 'env_nr', 'Wildlife': 'env_nr',
    'Environmental Quality': 'env_nr',
    'Environmental Response and Remediation': 'env_nr',
    'Department of Environmental Quality': 'env_nr',
    'Environment': 'env_nr', 'Natural Resources': 'env_nr',
    'Department of Natural Resources': 'env_nr',
    'Solid Waste': 'env_nr', 'Waste': 'env_nr',
    'Recycling': 'env_nr', 'Hazardous Materials': 'env_nr',
    'Hazardous Substances': 'env_nr',
    'Hazardous Materials Transportation': 'env_nr',
    'Underground Storage Tanks': 'env_nr',
    'Asbestos': 'env_nr', 'Emissions Control': 'env_nr',
    'Air': 'env_nr', 'Radiation': 'env_nr',
    'Pesticide & Pest Control': 'env_nr',
    'Pesticide and Pest Control': 'env_nr',
    'Reclamation': 'env_nr', 'Conservation District': 'env_nr',
    'Conservation Districts': 'env_nr',
    'State Lands': 'env_nr', 'Sovereign Lands': 'env_nr',
    'Trust Lands': 'env_nr', 'Public Lands': 'env_nr',
    'Geological Survey': 'env_nr', 'Oil and Gas': 'env_nr',
    'Fossil Fuels': 'env_nr', 'Natural Gas': 'env_nr',
    'Dams and Canals': 'env_nr', 'Weeds': 'env_nr',
    'Litter': 'env_nr', 'Noise Abatement': 'env_nr',
    'West Traverse Sentinel Landscape': 'env_nr',
    'Animal Health': 'env_nr', 'Animals': 'env_nr',

    # Transportation
    'Transportation': 'transpo', 'Motor Vehicles': 'transpo',
    'Roads/Highways': 'transpo', 'Highways': 'transpo',
    'Public Transit': 'transpo', 'Public Transit Districts': 'transpo',
    'Railroads': 'transpo', 'Bicycles': 'transpo',
    'Autonomous Vehicles': 'transpo', 'Aeronautics': 'transpo',
    'Airports': 'transpo', 'Boating': 'transpo',
    'Motor Carrier Regulation': 'transpo', 'Commercial Motor Vehicle Regulation': 'transpo',
    'Commercial Driver License (CDL)': 'transpo', 'Driver License': 'transpo',
    'Vehicle Registration': 'transpo', 'Motor Vehicle Insurance': 'transpo',
    'Motor Vehicle Equipment': 'transpo', 'Motor Vehicle Coverings': 'transpo',
    'Motor Vehicle Size Limits': 'transpo', 'Size Limits, Motor Vehicles': 'transpo',
    'Safety Inspection, Motor Vehicles': 'transpo', 'Vehicle Safety Equipment': 'transpo',
    'Off-highway Vehicles': 'transpo', 'Motorcycles': 'transpo',
    'Tow Trucks': 'transpo', 'Towing': 'transpo',
    'Traffic Violations': 'transpo', 'Driving Under the Influence (DUI)': 'transpo',
    'Seat Belt Laws': 'transpo', 'Seat Belts': 'transpo',
    'Helmets': 'transpo', 'Pedestrian Safety': 'transpo',
    'Speed Limits': 'transpo', 'Photo Radar': 'transpo',
    'Transportation Related Fees': 'transpo', 'Transportation Funding': 'transpo',
    'Transportation Fund': 'transpo', 'Transportation Commission': 'transpo',
    'Department of Transportation': 'transpo',
    'Transportation Investment Fund of 2005': 'transpo',
    'Transit Transportation Investment Fund': 'transpo',
    'Active Transportation Investment Fund': 'transpo',
    'Rural Transportation Investment Fund': 'transpo',
    'Active Transportation': 'transpo', 'Congestion Management': 'transpo',
    'Right of Way': 'transpo', 'License Plates': 'transpo',
    'Registration and Registration Fees': 'transpo',
    'Division of Motor Vehicles': 'transpo', 'Ports of Entry': 'transpo',
    'Unmanned Aircraft': 'transpo', 'Ridesharing and Shared Mobility': 'transpo',
    'Transportation Network Company': 'transpo',
    'Heber Valley Historic Railroad Authority': 'transpo',
    'Utah State Railroad Museum Authority': 'transpo',
    'Equipment, Motor Vehicles': 'transpo', 'Trails': 'transpo',

    # Elections & Government Operations
    'Elections': 'elect_gov', 'Election Law': 'elect_gov',
    'Election Administration': 'elect_gov', 'Voting and Voter Registration': 'elect_gov',
    'Voter Registration': 'elect_gov', 'Campaign Finance': 'elect_gov',
    'Redistricting': 'elect_gov', 'Political Parties': 'elect_gov',
    'Initiatives': 'elect_gov', 'Referenda': 'elect_gov',
    'Initiatives / Referendums': 'elect_gov', 'Initiatives \\ Referendums': 'elect_gov',
    'Legislative Operations': 'elect_gov', 'Legislative Organization': 'elect_gov',
    'Legislature': 'elect_gov', 'Legislative Staff Offices': 'elect_gov',
    'Legislative Affairs': 'elect_gov', 'Legislative Committees and Task Forces': 'elect_gov',
    'Committees, Legislative': 'elect_gov', 'Legislative Employees and Compensation': 'elect_gov',
    'Employees and Compensation, Legislative': 'elect_gov',
    'Legislative Publications': 'elect_gov', 'Legislative Oversight of Administrative Rules': 'elect_gov',
    'Legislative Water Development Commission': 'elect_gov',
    'Administrative Rules': 'elect_gov', 'Administrative Rulemaking': 'elect_gov',
    'Administrative Rulemaking and Procedures': 'elect_gov',
    'Administrative Law': 'elect_gov', 'Administrative Law Judges': 'elect_gov',
    'Administrative Procedures': 'elect_gov', 'Administrative Services': 'elect_gov',
    'Rulemaking Procedures': 'elect_gov', 'New Rulemaking Authority': 'elect_gov',
    'Office of Administrative Rules': 'elect_gov',
    'Judicial Review of Administrative Rules': 'elect_gov',
    'Judicial Review of State Agency Adjudicative Proceedings': 'elect_gov',
    'State Agency Adjudicative Proceedings': 'elect_gov',
    'State Agency Review of Adjudicative Proceedings': 'elect_gov',
    'Governor': 'elect_gov', 'Lieutenant Governor': 'elect_gov',
    'Attorney General': 'elect_gov', 'State Auditor': 'elect_gov',
    'State Treasurer': 'elect_gov', 'Constitutional Officers': 'elect_gov',
    'Elected Official': 'elect_gov', 'Public Officers': 'elect_gov',
    'County Officers': 'elect_gov', 'Municipal Officers': 'elect_gov',
    'Ethics': 'elect_gov', 'Conflicts of Interest': 'elect_gov',
    'Lobbying': 'elect_gov', 'Government Records': 'elect_gov',
    'Public Meetings': 'elect_gov', 'Open and Public Meetings': 'elect_gov',
    'Governance': 'elect_gov', 'Federalism': 'elect_gov',
    'Federal Government': 'elect_gov', 'Interlocal Cooperation': 'elect_gov',
    'Interlocal Agreements': 'elect_gov', 'Interlocal Entities': 'elect_gov',
    'Interlocal Cooperation Entities': 'elect_gov',
    'Task Force / Committees': 'elect_gov', 'Boards and Committees': 'elect_gov',
    'State Boards, Commissions, and Councils': 'elect_gov',
    'Revisor Legislation': 'elect_gov', 'Recodification': 'elect_gov',
    'Statutory Construction': 'elect_gov', 'Uniform Laws': 'elect_gov',
    'Resolutions': 'elect_gov', 'Resolutions, Rules': 'elect_gov',
    'Sunset Legislation': 'elect_gov', 'Sunsets and Repealers': 'elect_gov',
    'Boxcar Legislation': 'elect_gov', 'State Affairs in General': 'elect_gov',
    'Publications': 'elect_gov', 'Publications and Broadcasting': 'elect_gov',
    'Official Notices': 'elect_gov', 'Notice Requirements': 'elect_gov',
    'Archives': 'elect_gov', 'Archives and Records': 'elect_gov',
    'History': 'elect_gov', 'Capitol Hill': 'elect_gov',
    'State Symbols and Designations': 'elect_gov',
    'Flag (American)': 'elect_gov', 'Constitution': 'elect_gov',
    'Utah Constitutional Amendments': 'elect_gov',
    'Independent Entities': 'elect_gov',
    'Retirement and Independent State Entities': 'elect_gov','Governmental Immunity': 'elect_gov',
    'Tribal Nations Within Utah': 'elect_gov','Division of Indian Affairs': 'elect_gov',
    'Repatriation of Native American Remains': 'elect_gov',

    # Taxation & Finance
    'Income Tax': 'tax_fin', 'Individual Income Tax': 'tax_fin',
    'Sales Tax Exemptions': 'tax_fin', 'Sales and Use Tax': 'tax_fin',
    'Property Tax': 'tax_fin', 'Property Tax Collection': 'tax_fin',
    'Property Tax Relief': 'tax_fin', 'Farmland Assessment': 'tax_fin',
    'Assessment and Equalization': 'tax_fin', 'Truth in Taxation': 'tax_fin',
    'Tax Credits': 'tax_fin', 'Tax Increment Financing': 'tax_fin',
    'Excise Taxes': 'tax_fin', 'Corporate Tax': 'tax_fin',
    'Severance Tax': 'tax_fin', 'Privilege Tax': 'tax_fin',
    'Gross Receipts': 'tax_fin', 'Motor Fuel and Special Fuel Taxes': 'tax_fin',
    'Cigarette and Tobacco Taxes': 'tax_fin', 'Beer Tax': 'tax_fin',
    'Local Option Sales Taxes': 'tax_fin', 'Local Taxation and Fees': 'tax_fin',
    'Tribal Tax Issues': 'tax_fin', 'Revenue': 'tax_fin',
    'Revenue and Taxation': 'tax_fin', 'State Tax Commission': 'tax_fin',
    'Appropriations': 'tax_fin', 'Budgeting': 'tax_fin',
    'Public Budgeting': 'tax_fin', 'Fiscal Procedures': 'tax_fin',
    'Finance': 'tax_fin', 'Division of Finance': 'tax_fin',
    'Bonds': 'tax_fin', 'Public Bonds': 'tax_fin',
    'Public Funds and Accounts': 'tax_fin', 'Fees': 'tax_fin',
    'Uniform Fees': 'tax_fin', 'Fines': 'tax_fin',
    'Local Expenditures': 'tax_fin', 'Unclaimed Property': 'tax_fin',
    'Mineral Lease Funds': 'tax_fin', 'Permanent Community Impact Fund': 'tax_fin',
    'Impact Fees': 'tax_fin', 'Assessment Area': 'tax_fin',
    'Energy Assessment Area': 'tax_fin', 'County and Municipal Finance': 'tax_fin',
    'State Infrastructure Bank': 'tax_fin', 'Utah Capital Investment Corporation': 'tax_fin',
    'Risk Management Fund': 'tax_fin', 'Division of Risk Management': 'tax_fin',
    'Procurement': 'tax_fin', 'Government Purchasing': 'tax_fin',
    'Division of Purchasing and General Services': 'tax_fin',

    # Labor & Employment
    'Labor': 'labor_ee', 'Labor & Employment': 'labor_ee',
    'Labor and Employment': 'labor_ee', 'Labor Commission': 'labor_ee',
    'Labor Disputes': 'labor_ee', 'Labor Organizations': 'labor_ee',
    'Working Conditions': 'labor_ee', 'Workplace Safety and Health': 'labor_ee',
    'Occupational Safety and Health (OSHA)': 'labor_ee',
    'Occupational Disease': 'labor_ee', 'Industrial Accident': 'labor_ee',
    'Workers\' Compensation': 'labor_ee', 'Unemployment Insurance': 'labor_ee',
    'Wages': 'labor_ee', 'Right to Work': 'labor_ee',
    'Collective Bargaining': 'labor_ee', 'Unfair Labor Practices': 'labor_ee',
    'Employment Agencies': 'labor_ee', 'Employment of Minors': 'labor_ee',
    'Apprenticeship Training': 'labor_ee', 'Job Training': 'labor_ee',
    'Workforce Services': 'labor_ee', 'Department of Workforce Services': 'labor_ee',
    'Department of Human Resource Management': 'labor_ee',
    'Division of Human Resource Management': 'labor_ee',
    'State Officers and Employees': 'labor_ee',
    'Local Government Employees': 'labor_ee',
    'Sexual Harassment': 'labor_ee', 'Antidiscrimination': 'labor_ee',

    # Family & Social Services
    'Child Welfare': 'fam_ss', 'Foster Care': 'fam_ss',
    'Adoption': 'fam_ss', 'Child Custody/Parent Time': 'fam_ss',
    'Child Custody \\ Parent-Time': 'fam_ss',
    'Child Support': 'fam_ss', 'Child Care': 'fam_ss',
    'Child Care Licensing': 'fam_ss', 'Child \\ Day Care': 'fam_ss',
    'Child \\ Day Care Licensing': 'fam_ss',
    'Child Development': 'fam_ss', 'Children': 'fam_ss',
    'Minors': 'fam_ss', 'Juveniles': 'fam_ss',
    'Domestic Violence': 'fam_ss', 'Family': 'fam_ss',
    'Marriage / Divorce': 'fam_ss', 'Marriage/Divorce': 'fam_ss',
    'Parentage': 'fam_ss', 'Grandparents': 'fam_ss',
    'Parental Defense': 'fam_ss', 'Guardian Ad Litem': 'fam_ss',
    'Guardianship/Conservatorship': 'fam_ss', 'Guardians': 'fam_ss',
    'Conservators': 'fam_ss', 'Emancipation': 'fam_ss',
    'Homelessness': 'fam_ss', 'Homeless Persons': 'fam_ss',
    'Intergenerational Poverty': 'fam_ss', 'Public Assistance': 'fam_ss',
    'Human Services': 'fam_ss', 'Human Services Programs': 'fam_ss',
    'Human Services Licensure': 'fam_ss', 'Human Services Facilities': 'fam_ss',
    'Department of Human Services': 'fam_ss',
    'Adult Services': 'fam_ss', 'Elderly': 'fam_ss',
    'Division of Child and Family Services': 'fam_ss',
    'Office of Recovery Services': 'fam_ss',
    'Abuse, Neglect, Exploitation of Vulnerable Adults': 'fam_ss',
    'Abuse, Neglect, or Exploitation of Vulnerable Adults': 'fam_ss',
    'Abuse, Neglect, or Dependency': 'fam_ss',
    'Victims': 'fam_ss', 'Victims\' Rights': 'fam_ss',
    'Disabilities': 'fam_ss', 'Services for People with Disabilities': 'fam_ss',
    'Blind Persons': 'fam_ss', 'Blind and Visually-impaired Services': 'fam_ss',
    'Utah Schools for the Deaf and the Blind': 'fam_ss',
    'Deaf and Hard of Hearing': 'fam_ss',
    'Deaf and Hard of Hearing Services': 'fam_ss',
    'Refugees': 'fam_ss', 'Immigrants \\ Immigration': 'fam_ss',
    'Immigration': 'fam_ss', 'Indian Affairs': 'fam_ss',
    'Multicultural Affairs': 'fam_ss',
    'Food Security': 'fam_ss', 'Fatality Reports': 'fam_ss',
    'Death': 'fam_ss', 'Licensure, Human Services Programs': 'fam_ss',

    # Housing & Land Use
    'Affordable Housing': 'house_land_use', 'Housing': 'house_land_use',
    'Housing and Community Development': 'house_land_use',
    'Utah Housing Corporation': 'house_land_use', 'Planning and Zoning': 'house_land_use',
    'Real Estate': 'house_land_use', 'Real Property': 'house_land_use',
    'Landlord - Tenant': 'house_land_use', 'Landlord -- Tenant': 'house_land_use',
    'Land Use': 'house_land_use', 'Condominiums': 'house_land_use',
    'Community/Condominium Associations (HOAs)': 'house_land_use',
    'Eminent Domain': 'house_land_use',
    'Eminent Domain (Government Land Take Over)': 'house_land_use',
    'Annexation': 'house_land_use', 'Municipal Boundaries': 'house_land_use',
    'Local Government Boundaries': 'house_land_use',
    'Incorporation': 'house_land_use', 'Unincorporated Areas': 'house_land_use',
    'Fair Housing': 'house_land_use', 'Easements': 'house_land_use',
    'Property Conveyance': 'house_land_use', 'Mortgage': 'house_land_use',
    'Mortgage/Deed of Trust': 'house_land_use', 'Appraisals': 'house_land_use',
    'Title and Escrow': 'house_land_use', 'Liens': 'house_land_use',
    'Construction': 'house_land_use', 'Construction Industries': 'house_land_use',
    'Construction and Fire Codes': 'house_land_use',
    'State Buildings': 'house_land_use', 'State Facilities': 'house_land_use',
    'Facilities, Construction, and Management': 'house_land_use',
    'Local Building Authority': 'house_land_use',
    'Community Development': 'house_land_use',
    'Community Development and Renewal Agencies': 'house_land_use',
    'Community Reinvestment Agencies': 'house_land_use',

    # Business & Commerce
    'Business': 'bus_comm', 'Commerce and Trade': 'bus_comm',
    'commerce': 'bus_comm', 'Corporations': 'bus_comm',
    'Limited Liability Company': 'bus_comm', 'Limited Liability Companies': 'bus_comm',
    'Partnerships': 'bus_comm', 'Business Entities': 'bus_comm',
    'Business Licensing': 'bus_comm', 'Business Development': 'bus_comm',
    'Contracts': 'bus_comm', 'Contracts and Obligations': 'bus_comm',
    'Consumer Protection': 'bus_comm', 'Consumer Credit': 'bus_comm',
    'Credit': 'bus_comm', 'Antitrust': 'bus_comm',
    'Antitrust Law': 'bus_comm', 'Insurance': 'bus_comm',
    'Insurance Department': 'bus_comm', 'Department of Insurance': 'bus_comm',
    'Property and Casualty Insurance': 'bus_comm',
    'Life Insurance and Annuities': 'bus_comm',
    'Motor Vehicle Insurance': 'bus_comm',
    'Occupational and Professional Licensing': 'bus_comm',
    'Division of Professional Licensing': 'bus_comm',
    'Office of Licensing': 'bus_comm',
    'Office of Licensing and Background Checks': 'bus_comm',
    'Occupations and Professions': 'bus_comm',
    'Securities': 'bus_comm', 'Bankruptcy': 'bus_comm',
    'Franchise': 'bus_comm', 'Franchises': 'bus_comm',
    'Retail Trade': 'bus_comm', 'Manufacturing': 'bus_comm',
    'Advertising': 'bus_comm', 'Trademarks': 'bus_comm',
    'International Trade': 'bus_comm', 'Tourism': 'bus_comm',
    'Department of Commerce': 'bus_comm',
    'Department of Financial Institutions': 'bus_comm',
    'Financial Institutions': 'bus_comm', 'Banks': 'bus_comm',
    'Credit Unions': 'bus_comm', 'Charities': 'bus_comm',
    'Governmental Nonprofit Corporations': 'bus_comm',
    'Collection Agencies': 'bus_comm', 'Pawnshops and Secondhand Merchandise': 'bus_comm',
    'Rental and Leasing Services': 'bus_comm',
    'Repair and Maintenance Services': 'bus_comm',
    'Food Services and Drinking Places': 'bus_comm',
    'Hotels and Hotel Keepers': 'bus_comm', 'Warehousing and Storage': 'bus_comm',
    'Personal Services': 'bus_comm', 'Process Servers': 'bus_comm',
    'Security Licensing': 'bus_comm', 'Uniform Commercial Code': 'bus_comm',
    'Electronic Transactions': 'bus_comm', 'Negotiable Certificates': 'bus_comm',
    'Product Liability': 'bus_comm', 'Liability': 'bus_comm',

    # Public Safety & Emergency Management
    'Public Safety': 'pub_safe_emerg', 'Fire Protection': 'pub_safe_emerg',
    'Fire Control': 'pub_safe_emerg', 'Emergency Management': 'pub_safe_emerg',
    'Emergency Powers': 'pub_safe_emerg', 'Homeland Security': 'pub_safe_emerg',
    'National Guard': 'pub_safe_emerg', 'Utah National Guard': 'pub_safe_emerg',
    'Military Services': 'pub_safe_emerg',
    'Department of Public Safety': 'pub_safe_emerg',
    'Veterans and Military Benefits': 'pub_safe_emerg',
    'Veterans and Military Affairs': 'pub_safe_emerg',
    'Veterans Affairs': 'pub_safe_emerg', 'Veterans\' Affairs': 'pub_safe_emerg',
    'Veterans Service Organizations': 'pub_safe_emerg',
    'Utah Veterans Advisory Council': 'pub_safe_emerg',
    'Department of Veterans and Military Affairs': 'pub_safe_emerg',
    'Military Installation Development Authority': 'pub_safe_emerg',
    'Miners': 'pub_safe_emerg', 'Background Checks': 'pub_safe_emerg',
    'Crisis Line': 'pub_safe_emerg',
    'Public Safety Retirement': 'pub_safe_emerg',
    'Firefighters\' Retirement': 'pub_safe_emerg',

    # Technology & Data
    'Technology': 'tech_data', 'Technology Governance': 'tech_data',
    'Department of Technology Services': 'tech_data',
    'Division of Technology Services': 'tech_data',
    'Agency Technology Services': 'tech_data',
    'Agency Technology Services and Public Websites': 'tech_data',
    'Information Technology Rate Committee': 'tech_data',
    'Artificial Intelligence': 'tech_data', 'Data and Cyber Security': 'tech_data',
    'Electronic Privacy': 'tech_data', 'Electronic Information': 'tech_data',
    'Electronic Databases': 'tech_data', 'Internet': 'tech_data',
    'Internet Protocols': 'tech_data', 'Internet Service Provider': 'tech_data',
    'Broadband': 'tech_data', 'Telecommunications': 'tech_data',
    'Telephone': 'tech_data', 'Social Media': 'tech_data',
    'Blockchain': 'tech_data', 'Management Information System': 'tech_data',
    'Automated Geographic Reference Center': 'tech_data',
    'Utah Geospatial Resource Center': 'tech_data',
    'Recordings': 'tech_data', 'News Agencies': 'tech_data',

    # Civil Rights & Liberties
    'Civil Rights': 'civ_rights_libs', 'Antidiscrimination': 'civ_rights_libs',
    'Fair Housing': 'civ_rights_libs', 'Sexual Harassment': 'civ_rights_libs',
    'Pornography': 'civ_rights_libs',
    'Freedom of Speech - Const. Art. I, Sec 15': 'civ_rights_libs',
    'On-campus Speech': 'civ_rights_libs', 'Civil Litigation': 'civ_rights_libs',
    'Civil Procedure': 'civ_rights_libs', 'Statute of Limitations': 'civ_rights_libs',
    'Subpoenas': 'civ_rights_libs', 'Juries': 'civ_rights_libs',
    'Appeals': 'civ_rights_libs', 'Court Rules': 'civ_rights_libs',
    'Court Procedure': 'civ_rights_libs', 'Judicial Administration': 'civ_rights_libs',
    'Judicial Operations': 'civ_rights_libs', 'Judiciary': 'civ_rights_libs',
    'Judges': 'civ_rights_libs', 'Attorneys': 'civ_rights_libs',
    'Alternative Dispute Resolution': 'civ_rights_libs',
    'Arbitration\\Mediation': 'civ_rights_libs',
    'Dispute Resolution': 'civ_rights_libs',
    'Notarization and Authentication': 'civ_rights_libs',

    # Energy
    'Energy': 'energy', 'Renewable Energy': 'energy', 'Renewable and Clean Energy': 'energy',
    'Clean Energy': 'energy', 'Clean Fuels': 'energy', 'Solar Power': 'energy',
    'Wind Power': 'energy', 'Nuclear Energy': 'energy', 'Thermal Power': 'energy',
    'Electricity': 'energy', 'Electric Cooperatives': 'energy',
    'Investor-owned Utilities': 'energy', 'Public Utilities': 'energy',
    'Public Utilities & Technology': 'energy', 'Public Utilities and Technology': 'energy',
    'Public Service Commission': 'energy', 'Division of Public Utilities': 'energy',
    'Utilities Siting': 'energy', 'Utilities Siting and Permitting': 'energy',
    'Utility Programs and Energy Procurement': 'energy',
    'Utah Energy Infrastructure Authority': 'energy',
    'Office of Energy Development': 'energy',
    'Governor\'s Office of Energy Development': 'energy', 'Energy Storage': 'energy',

    # Local Government
    'Counties': 'local_gov', 'Municipalities': 'local_gov',
    'Municipal Government': 'local_gov', 'County Government': 'local_gov',
    'Local Governments': 'local_gov', 'Local Government Ordinances': 'local_gov',
    'Local Government Infrastructure and Utilities': 'local_gov',
    'Local Government Controlled Districts': 'local_gov',
    'Local Governments Controlled Districts': 'local_gov',
    'Local Government Classification': 'local_gov',
    'Local Government Employees': 'local_gov',
    'Special Service District': 'local_gov', 'Special Service Districts': 'local_gov',
    'Limited Purpose Local Government Entities': 'local_gov',
    'Local Districts': 'local_gov', 'Metro Townships': 'local_gov',
    'Townships': 'local_gov', 'Form of Local Government': 'local_gov',
    'Community Councils': 'local_gov', 'Political Subdivisions (Local Issues)': 'local_gov',
    'Public Infrastructure District': 'local_gov',
    'Rural Development': 'local_gov',

    # Economic Development
    'Economic Development': 'econ_dev',
    'Office of Economic Development': 'econ_dev',
    'Governor\'s Office of Economic Development': 'econ_dev',
    'Governor\'s Office of Economic Opportunity': 'econ_dev',
    'Business Development': 'econ_dev', 'Science and Biotechnology': 'econ_dev',
    'Olympics': 'econ_dev', 'Compacts': 'econ_dev',
    'International Trade': 'econ_dev',

    # Retirement & Benefits
    'Retirement': 'retire_ben', 'Utah Retirement Systems (URS)': 'retire_ben',
    'Public Employees Retirement': 'retire_ben',
    'Tier 1 Retirement': 'retire_ben', 'Tier 2 Retirement': 'retire_ben',
    'Constitutional Officer and Legislator Retirement': 'retire_ben',
    'Judges\' Retirement': 'retire_ben',
    'Public Safety Retirement': 'retire_ben',
    'Private Employer Retirement': 'retire_ben',
    'Public Employees Long-term Disabilities Benefits': 'retire_ben',
    'Public Employees Insurance and Benefits': 'retire_ben',
    'Public Retirement and Insurance': 'retire_ben',
    'Veterans and Military Benefits': 'retire_ben',

    # Agriculture & Food
    'Agriculture': 'agri_food', 'Agriculture and Food': 'agri_food',
    'Agriculture & Food': 'agri_food', 'Department of Agriculture and Food': 'agri_food',
    'Agriculture Inspections': 'agri_food', 'Livestock': 'agri_food',
    'Animal Food Products': 'agri_food', 'Plant Food Products': 'agri_food',
    'Plant Industry': 'agri_food', 'Food': 'agri_food',
    'Food Quality': 'agri_food', 'Dairy': 'agri_food',
    'Utah Dairy Commission': 'agri_food', 'Aquaculture and Game Ranching': 'agri_food',
    'Game Ranching': 'agri_food', 'Fishing': 'agri_food',
    'Hunting': 'agri_food', 'Veterinary Care': 'agri_food',
    'Dogs': 'agri_food',

    # Arts, Culture & Recreation
    'Arts': 'arts_cult_rec', 'Heritage': 'arts_cult_rec',
    'Heritage and Arts': 'arts_cult_rec',
    'Department of Heritage and Arts': 'arts_cult_rec',
    'Department of Cultural and Community Engagement': 'arts_cult_rec',
    'Cultural and Community Engagement': 'arts_cult_rec',
    'Cultural Engagement': 'arts_cult_rec', 'Museums': 'arts_cult_rec',
    'Libraries': 'arts_cult_rec', 'Recreation': 'arts_cult_rec',
    'Parks and Recreation': 'arts_cult_rec',
    'State Parks and Recreation': 'arts_cult_rec',
    'Division of State Parks': 'arts_cult_rec',
    'Division of Recreation': 'arts_cult_rec',
    'Outdoor Recreation': 'arts_cult_rec', 'Parks': 'arts_cult_rec',
    'Olympics': 'arts_cult_rec', 'Athletics': 'arts_cult_rec',
    'Motion Pictures': 'arts_cult_rec',

    # No Subjects
    'None': 'no_subject',

}

# one hot code subject groups
def get_all_groups(subject_object):
    if pd.isna(subject_object):
        return []
    subjects = [sub.strip() for sub in subject_object.split('; ')]
    groups = set(subject_to_group.get(subj) for subj in subjects if subj in subject_to_group)
    return list(groups)

df_bills['subject_groups'] = df_bills['subjects'].apply(get_all_groups)

from sklearn.preprocessing import MultiLabelBinarizer
mlb = MultiLabelBinarizer()

group_dummies = pd.DataFrame(
    mlb.fit_transform(df_bills['subject_groups']),
    columns=[g for g in mlb.classes_],
    index=df_bills.index
)

df_bills = pd.concat([df_bills, group_dummies], axis=1) # ONLY RUN THIS ONCE !!



# making one-hot dummy variables (committee, cosponsor, floor sponsor, money appropriated
dummy_mask = ['committees','co_sponsors', 'floor_sponsor', 'money_appropriated']
dummy_name = ['committee1', 'cospon1', 'floorspon1', 'money1']
df_bills[dummy_name] = df_bills[dummy_mask].map(lambda x: 0 if x == 'None' else 1)




# get # of cosponsors variable
def cospon_sum(subject_object):
    if pd.isna(subject_object) or subject_object == 'None':
        return 0
    cospon = [c for c in subject_object.split('; ')]
    return len(cospon)

df_bills['cospon_num'] = df_bills['co_sponsors'].apply(cospon_sum)




# update 'num_substitutes' to align with active version (pre-2025); new column 'num_sub_new'
df_bills['num_sub1'] = np.where(
    df_bills['session_year'] < 2025,
    df_bills['active_version'],
    df_bills['num_substitutes'] )
df_bills.tail()

def sub_num(subject_object):
    if pd.isna(subject_object):
        return 0
    if isinstance(subject_object, int):
        return subject_object
    if subject_object[-3] == 'S':
        return int(subject_object[-1])
    else:
        return 0

df_bills['num_sub_new'] = df_bills['num_sub1'].apply(sub_num)
df_bills = df_bills.drop(columns=['num_sub1']) # this was a temporary column; i dont want to store it LOL




# extracting and summing money amounts
def money_sub(money_big):
    if pd.isna(money_big):
        return 0
    if money_big == 'None':
        return 0
    cells = [cell.strip() for cell in money_big.split('<hr>')]
    money_values = []
    for cell in cells:
        match = re.search(r'\$([0-9,]+)', cell)
        if match:
            money_values.append(int(re.search(r'\$([0-9,]+)', cell).group(1).replace(',','')))
            return sum(money_values)
        else:
            return 0

df_bills['money_sum'] = df_bills['money_appropriated'].apply(money_sub)



print(df_bills.shape)
df_bills.head()